# Ad-hoc Inference with Delta Shared Models

This notebook demonstrates the full flow of consuming a shared model via Delta Sharing and utilizing it for ad-hoc inference.

Steps covered:

1. **Accept the shared Delta Share**  
   - Programmatically create a catalog from the shared model using Delta Sharing APIs.

2. **Ad-hoc Inference from Model Registry**  
   - Load the shared model directly from the Unity Catalog Model Registry.
   - Perform batch inference using the model within the cluster, assuming cluster environment compatibility.


In [0]:
%sql
-- Step 0: Accept Delta Share (Run once using SQL)
-- databricks-field-eng has already been configured as a Delta Share Provider
CREATE CATALOG IF NOT EXISTS abooth_delta_share
USING SHARE `databricks-field-eng`.`abooth_bike_sharing_model_share`
COMMENT "Catalog for accessing shared bike sharing model.";


## Load and Use the Shared Model (Ad-hoc Inference)

In this section:
- Configure MLflow to use Unity Catalog as the model registry.
- Load the shared models using the model URI and alias (`@prod`, `@staging`).
- Prepare a small sample of new data that matches the training schema.
- Perform ad-hoc predictions directly on the cluster without deploying a separate endpoint.

This demonstrates how quickly models shared via Delta Sharing can be used for immediate inference inside a recipient workspace.


In [0]:
# Step 1: Load Model from Unity Catalog (Recipient Workspace)
import mlflow
import pandas as pd

# Configure MLflow to use Databricks Unity Catalog
mlflow.set_registry_uri("databricks-uc")

In [0]:
# Load the Champion model using the shared catalog
model_uri = "models:/abooth_delta_share.default.bike_sharing_uc_model@prod"
loaded_model = mlflow.sklearn.load_model(model_uri)

# Step 2: Prepare New Data
new_data = pd.DataFrame({
    'season': [1],
    'yr': [0],
    'mnth': [1],
    'hr': [10],
    'holiday': [0],
    'weekday': [3],
    'workingday': [1],
    'weathersit': [1],
    'temp': [0.3],
    'atemp': [0.31],
    'hum': [0.5],
    'windspeed': [0.1]
})

# Step 3: Make Predictions
prediction = loaded_model.predict(new_data)
print(f"Predicted bike rentals: {prediction[0]}")


/databricks/python/lib/python3.12/site-packages/databricks/sdk/service/jobs.py:60: SyntaxWarning: invalid escape sequence '\.'
  """The sequence number of this run attempt for a triggered job run. The initial attempt of a run
/databricks/python/lib/python3.12/site-packages/databricks/sdk/service/jobs.py:2570: SyntaxWarning: invalid escape sequence '\.'
  """The sequence number of this run attempt for a triggered job run. The initial attempt of a run
/databricks/python/lib/python3.12/site-packages/databricks/sdk/service/jobs.py:3431: SyntaxWarning: invalid escape sequence '\.'
  """The sequence number of this run attempt for a triggered job run. The initial attempt of a run


[LightGBM] [Warning] lambda_l2 is set=1.8264397243517438, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.8264397243517438
[LightGBM] [Warning] lambda_l1 is set=0.1578384774673136, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1578384774673136
Predicted bike rentals: 20.787938243332736


In [0]:
# Load the Challenger model using the shared catalog
model_uri = "models:/abooth_delta_share.default.bike_sharing_uc_model@staging"
loaded_model = mlflow.sklearn.load_model(model_uri)

# Step 2: Prepare New Data
new_data = pd.DataFrame({
    'season': [1],
    'yr': [0],
    'mnth': [1],
    'hr': [10],
    'holiday': [0],
    'weekday': [3],
    'workingday': [1],
    'weathersit': [1],
    'temp': [0.3],
    'atemp': [0.31],
    'hum': [0.5],
    'windspeed': [0.1]
})

# Step 3: Make Predictions
prediction = loaded_model.predict(new_data)
print(f"Predicted bike rentals: {prediction[0]}")


Predicted bike rentals: 40.205448150634766


## ✅ Next Steps

In the next section, we will deploy the shared and registered models as real-time serving endpoints using Databricks Model Serving. This will enable us to evaluate the models through A/B testing.

Steps:
- Programmatically create new serving endpoints using the Databricks REST API.
- Configure each endpoint to serve a shared model version from Unity Catalog.
- Enable auto-scaling options, such as `scale_to_zero`, for cost efficiency.

Once the endpoints are active, we will send inference requests over HTTP by authenticating with a Databricks personal access token (PAT). This approach enables scalable, production-grade, low-latency predictions directly from the shared models.

